# 📊 01 — Exploratory Data Analysis

This notebook demonstrates the full EDA pipeline using the Survey ML Toolkit.

**Objectives:**
- Generate or load survey data
- Detect column types automatically
- Clean data (speeders, straightliners, missing values)
- Explore distributions, correlations, and patterns
- Generate publication-ready visualizations

---

## 1. Setup & Imports

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from survey_toolkit import (
    SurveyLoader,
    SurveyCleaner,
    SurveyEDA,
    generate_sample_survey,
    detect_column_types,
    validate_survey_data,
)

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✅ Setup complete")

## 2. Generate Sample Data

We'll generate a realistic survey dataset with:
- 500 respondents
- 10 Likert-scale items (1–5)
- Demographic variables
- Duration, NPS, satisfaction groups
- Open-ended responses
- Realistic data quality issues (speeders, straightliners, missing data)

In [ ]:
df = generate_sample_survey(
    n_respondents=500,
    n_likert_items=10,
    n_demographic_cols=4,
    include_open_ended=True,
    random_state=42,
    save_path="../data/sample_survey.csv",
)

print(f"📋 Dataset shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")
df.head()

## 3. Data Validation

Before diving in, let's validate the data quality.

In [ ]:
validation = validate_survey_data(
    df,
    required_columns=["respondent_id", "q1", "q2", "age_group"],
    max_missing_pct=50,
    min_respondents=30,
)

print(f"✅ Valid: {validation['valid']}")
print(f"📊 Respondents: {validation['n_respondents']}")
print(f"📊 Columns: {validation['n_columns']}")
print(f"📊 Overall missing: {validation['overall_missing_pct']}%")

if validation["issues"]:
    print(f"\n❌ Issues:")
    for issue in validation["issues"]:
        print(f"   - {issue}")

if validation["warnings"]:
    print(f"\n⚠️  Warnings:")
    for warning in validation["warnings"]:
        print(f"   - {warning}")

## 4. Automatic Column Type Detection

The toolkit can automatically identify Likert items, demographics,
IDs, and free-text columns.

In [ ]:
column_types = detect_column_types(df)

print("🔍 Detected Column Types:")
print("=" * 50)
for col_type, columns in column_types.items():
    if columns:
        print(f"\n{col_type.upper()} ({len(columns)} columns):")
        for col in columns:
            print(f"   - {col}")

In [ ]:
# Store for later use
likert_cols = column_types["likert"]
demographic_cols = column_types["categorical"]
id_cols = column_types["id"]
text_cols = column_types["text"]

print(f"\n📊 Likert columns ({len(likert_cols)}): {likert_cols}")
print(f"👥 Demographic columns ({len(demographic_cols)}): {demographic_cols}")

## 5. Data Cleaning

Apply survey-specific cleaning steps:
1. **Remove speeders** — respondents who finished too fast
2. **Remove straightliners** — respondents who gave the same answer for everything
3. **Handle missing data** — fill with median for numeric columns

In [ ]:
# Check raw data quality
print("📋 BEFORE CLEANING")
print(f"   Total respondents: {len(df)}")
print(f"   Speeders (< 60s): {(df['duration_seconds'] < 60).sum()}")
print(f"   Missing values: {df[likert_cols].isna().sum().sum()}")
print(f"   Duration range: {df['duration_seconds'].min()}s – {df['duration_seconds'].max()}s")

In [ ]:
cleaner = SurveyCleaner(df)
clean_df = (
    cleaner
    .remove_speeders("duration_seconds", min_seconds=60)
    .remove_straightliners(likert_cols, threshold=0.95)
    .handle_missing(strategy="median", columns=likert_cols)
    .remove_duplicates()
    .get_clean_data()
)

print("\n📋 AFTER CLEANING")
print(f"   Total respondents: {len(clean_df)}")
print(f"   Missing values: {clean_df[likert_cols].isna().sum().sum()}")
print(f"   Duration range: {clean_df['duration_seconds'].min()}s – {clean_df['duration_seconds'].max()}s")

In [ ]:
# Review cleaning log
print("\n📝 Cleaning Log:")
for step in cleaner.get_log():
    print(f"   [{step['action']}] {step['details']}")

## 6. Exploratory Data Analysis

### 6.1 Response Summary Table

In [ ]:
eda = SurveyEDA(clean_df, output_dir="../outputs/figures")
summary = eda.response_summary()
summary

### 6.2 Likert Scale Distributions

In [ ]:
eda.plot_likert_distribution(
    likert_cols,
    labels={f"q{i}": f"Question {i}" for i in range(1, 11)},
    save=True,
)

### 6.3 Individual Likert Item Histograms

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
axes = axes.flatten()

for i, col in enumerate(likert_cols):
    ax = axes[i]
    counts = clean_df[col].value_counts().sort_index()
    colors = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]
    ax.bar(counts.index, counts.values, color=colors, edgecolor="white")
    ax.set_title(f"Q{i+1}", fontsize=12)
    ax.set_xlabel("")
    ax.set_xticks([1, 2, 3, 4, 5])
    if i % 5 == 0:
        ax.set_ylabel("Count")

plt.suptitle("Response Distribution per Question", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/likert_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.4 Demographic Breakdowns

In [ ]:
for col in demographic_cols[:3]:
    eda.plot_demographic_breakdown(col, plot_type="bar", save=True)

### 6.5 Responses by Demographic Group

In [ ]:
eda.plot_response_by_group("q1", "age_group", plot_type="box", save=True)

In [ ]:
eda.plot_response_by_group("q1", "age_group", plot_type="violin", save=True)

### 6.6 Correlation Heatmap

In [ ]:
eda.plot_correlation_heatmap(likert_cols, method="pearson", save=True)

### 6.7 Missing Data Analysis

In [ ]:
eda.missing_data_report(save=True)

### 6.8 NPS Distribution

In [ ]:
if "nps_score" in clean_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))

    nps = clean_df["nps_score"]
    colors = []
    for val in range(11):
        if val <= 6:
            colors.append("#d73027")   # Detractor
        elif val <= 8:
            colors.append("#fee08b")   # Passive
        else:
            colors.append("#1a9850")   # Promoter

    counts = nps.value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=[colors[int(i)] for i in counts.index],
           edgecolor="white")
    ax.set_xlabel("NPS Score")
    ax.set_ylabel("Count")
    ax.set_title("Net Promoter Score Distribution")
    ax.set_xticks(range(11))

    # Calculate NPS
    promoters = (nps >= 9).sum() / len(nps) * 100
    passives = ((nps >= 7) & (nps <= 8)).sum() / len(nps) * 100
    detractors = (nps <= 6).sum() / len(nps) * 100
    nps_score = promoters - detractors

    ax.text(0.98, 0.95,
            f"NPS: {nps_score:.0f}\n"
            f"Promoters: {promoters:.1f}%\n"
            f"Passives: {passives:.1f}%\n"
            f"Detractors: {detractors:.1f}%",
            transform=ax.transAxes, va="top", ha="right",
            fontsize=10, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

    plt.tight_layout()
    plt.savefig("../outputs/figures/nps_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

### 6.9 Satisfaction Group Overview

In [ ]:
if "satisfaction_group" in clean_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart
    sat_counts = clean_df["satisfaction_group"].value_counts()
    colors_sat = {"Satisfied": "#1a9850", "Neutral": "#fee08b", "Dissatisfied": "#d73027"}
    sat_colors = [colors_sat.get(label, "gray") for label in sat_counts.index]

    axes[0].pie(
        sat_counts, labels=sat_counts.index, autopct="%1.1f%%",
        colors=sat_colors, startangle=90, textprops={"fontsize": 11}
    )
    axes[0].set_title("Satisfaction Group Distribution")

    # Mean Likert score by satisfaction group
    group_means = clean_df.groupby("satisfaction_group")[likert_cols].mean().mean(axis=1)
    group_means = group_means.reindex(["Dissatisfied", "Neutral", "Satisfied"])
    bar_colors = [colors_sat[g] for g in group_means.index]
    axes[1].bar(group_means.index, group_means.values, color=bar_colors, edgecolor="white")
    axes[1].set_ylabel("Mean Likert Score")
    axes[1].set_title("Average Score by Satisfaction Group")
    axes[1].set_ylim(1, 5)

    plt.tight_layout()
    plt.savefig("../outputs/figures/satisfaction_overview.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Summary

| Metric | Value |
|--------|-------|
| Total respondents (raw) | 500 |
| After cleaning | {len(clean_df)} |
| Likert items | {len(likert_cols)} |
| Missing data remaining | {clean_df[likert_cols].isna().sum().sum()} |
| Figures generated | {len(eda.figures)} |

In [ ]:
print(f"📊 EDA Complete!")
print(f"   Respondents analyzed: {len(clean_df)}")
print(f"   Figures generated: {len(eda.figures)}")
print(f"   Saved to: ../outputs/figures/")
print(f"\n   Files:")
for fig_path in eda.figures:
    print(f"     - {fig_path}")